# **06 - PhoBERT cho Phân loại Bình luận Tiếng Việt**

Notebook này thực hiện huấn luyện mô hình PhoBERT-base-v2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## **Phần 0: Cài đặt và Import**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install transformers datasets accelerate huggingface_hub torch torchvision torchaudio

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

_cwd = Path(".").resolve()
if (_cwd / "data").exists():
    ROOT_DIR = _cwd
elif (_cwd.parent / "data").exists():
    ROOT_DIR = _cwd.parent
elif "ROOT_DIR" not in locals() and "ROOT_DIR" not in globals():
    ROOT_DIR = Path("/content/drive/MyDrive/Intro2MLFinal") if Path("/content/drive").exists() else _cwd
    print(f"Cảnh báo: Không tìm thấy thư mục data. Tạm gán ROOT_DIR = {ROOT_DIR}")

DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "output" / "phobert"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {'GPU ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

Cảnh báo: Không tìm thấy thư mục data. Tạm gán ROOT_DIR = /content/drive/MyDrive/Intro2MLFinal
Device: GPU Tesla T4


---
## **Phần 1: Chuẩn hóa Teencode & Đọc Dữ liệu**

In [ ]:
TEENCODE_DICT = {
    "dell": "đéo", "del": "đéo", "đell": "đéo", "đel": "đéo",
    "loz": "lồn", "lon": "lồn", "lòn": "lồn", "l": "lồn",
    "coin card": "củ cặc", "cc": "củ cặc", "cức": "cứt",
    "ms": "mới", "bh": "bây giờ", "kb": "không biết",
    "kk": "cười", "haha": "cười", "đhs": "đéo hiểu sao",
    "dm": "địt mẹ", "đm": "địt mẹ", "dmm": "địt mẹ mày", "vcl": "vãi lồn",
    "vl": "vãi lồn", "vkl": "vãi lồn", "cl": "cái lồn", "clgt": "cái lồn gì thế",
    "đcm": "địt con mẹ", "dcm": "địt con mẹ"
}

def normalize_teencode(text: str) -> str:
    words = str(text).split()
    return " ".join([TEENCODE_DICT.get(w, w) for w in words])

def load_data(split):
    df = pd.read_csv(DATA_DIR / f"{split}_clean.csv")
    df["free_text_clean"] = df["free_text_clean"].apply(normalize_teencode)
    return Dataset.from_pandas(df[["free_text_clean", "label_id"]].rename(columns={"label_id": "label"}))

train_ds = load_data("train")
dev_ds = load_data("dev")
test_ds = load_data("test")

print(f"Train: {len(train_ds)} | Dev: {len(dev_ds)} | Test: {len(test_ds)}")

Train: 22510 | Dev: 2633 | Test: 6527


---
## **Phần 2: Tokenization (PhoBERT)**

In [ ]:
MODEL_NAME = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["free_text_clean"], padding="max_length", truncation=True, max_length=256)

train_encoded = train_ds.map(tokenize_function, batched=True)
dev_encoded = dev_ds.map(tokenize_function, batched=True)
test_encoded = test_ds.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

Map:   0%|          | 0/22510 [00:00<?, ? examples/s]

Map:   0%|          | 0/2633 [00:00<?, ? examples/s]

Map:   0%|          | 0/6527 [00:00<?, ? examples/s]

---
## **Phần 3: Cấu hình Hàm Loss (Xử lý Imbalance)**

Tính toán trọng số lớp (Class Weights) dựa trên phân phối nhãn của tập train để trừng phạt mạnh mô hình khi dự đoán sai lớp thiểu số (HATE, OFFENSIVE).

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from torch import nn

labels = train_ds["label"]
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

if torch.cuda.is_available():
    class_weights_tensor = class_weights_tensor.cuda()

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

---
## **Phần 4: Huấn luyện PhoBERT (Tối ưu 4GB VRAM)**

Dùng batch_size=4 và gradient_accumulation_steps=4 để chạy mượt mà trên GPU 4GB.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weighted}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=dev_encoded,
    compute_metrics=compute_metrics,
)

trainer.train()

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.748235,0.666760,0.804406,0.622735,0.826859
2,0.612513,0.699946,0.847322,0.683541,0.858565
3,0.487051,0.790567,0.855298,0.684407,0.862800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=4221, training_loss=0.6289814052297108, metrics={'train_runtime': 1032.0168, 'train_samples_per_second': 65.435, 'train_steps_per_second': 4.09, 'total_flos': 8884024549585920.0, 'train_loss': 0.6289814052297108, 'epoch': 3.0})

---
## **Phần 5: Đánh giá & Lưu Kết quả**

In [ ]:
test_results = trainer.predict(test_encoded)
print(test_results.metrics)

preds = test_results.predictions.argmax(-1)
report = classification_report(test_ds["label"], preds, target_names=["CLEAN", "OFFENSIVE", "HATE"])
print("\nClassification Report:\n", report)

trainer.save_model(OUTPUT_DIR / "phobert_best")
tokenizer.save_pretrained(OUTPUT_DIR / "phobert_best")
print("Saved best model to output/phobert/phobert_best")

{'test_loss': 0.8986178040504456, 'test_accuracy': 0.8558296307645166, 'test_f1_macro': 0.6703476564759846, 'test_f1_weighted': 0.8636920542929414, 'test_runtime': 24.5884, 'test_samples_per_second': 265.45, 'test_steps_per_second': 8.297}

Classification Report:
               precision    recall  f1-score   support

       CLEAN       0.95      0.90      0.93      5406
   OFFENSIVE       0.40      0.52      0.45       440
        HATE       0.58      0.70      0.63       681

    accuracy                           0.86      6527
   macro avg       0.64      0.71      0.67      6527
weighted avg       0.87      0.86      0.86      6527



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model to output/phobert/phobert_best
